# OrbitLab: Mathematical Orbital Simulator
This notebook provides a robust framework for simulating N-body gravitational systems, ranging from Earth-Moon dynamics to binary stars and complex multi-body orbits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List

G = 6.67430e-11

@dataclass
class Body:
    name: str
    mass: float
    position: np.ndarray
    velocity: np.ndarray
    color: str = 'blue'

class OrbitSimulator:
    def __init__(self, bodies: List[Body]):
        self.bodies = bodies
        self.history = {b.name: {'pos': [], 'vel': [], 'energy': []} for b in bodies}

    def compute_accelerations(self, positions):
        n = len(self.bodies)
        acc = np.zeros((n, 3))
        for i in range(n):
            for j in range(i + 1, n):
                r_vec = positions[j] - positions[i]
                dist = np.linalg.norm(r_vec)
                force_mag = G * self.bodies[i].mass * self.bodies[j].mass / (dist**3)
                acc[i] += force_mag * r_vec / self.bodies[i].mass
                acc[j] -= force_mag * r_vec / self.bodies[j].mass
        return acc

    def step(self, dt):
        pos = np.array([b.position for b in self.bodies])
        vel = np.array([b.velocity for b in self.bodies])

        def get_derivs(p, v):
            a = self.compute_accelerations(p)
            return v, a

        kv1, ka1 = get_derivs(pos, vel)
        kv2, ka2 = get_derivs(pos + kv1*dt/2, vel + ka1*dt/2)
        kv3, ka3 = get_derivs(pos + kv2*dt/2, vel + ka2*dt/2)
        kv4, ka4 = get_derivs(pos + kv3*dt, vel + ka3*dt)

        new_pos = pos + (dt/6) * (kv1 + 2*kv2 + 2*kv3 + kv4)
        new_vel = vel + (dt/6) * (ka1 + 2*ka2 + 2*ka3 + ka4)

        for i, b in enumerate(self.bodies):
            b.position = new_pos[i]
            b.velocity = new_vel[i]
            self.history[b.name]['pos'].append(b.position.copy())
            self.history[b.name]['vel'].append(np.linalg.norm(b.velocity))

    def run(self, duration, dt):
        steps = int(duration / dt)
        for _ in range(steps):
            self.step(dt)

## Earth-Moon System Simulation
Let's set up a simulation of the Earth and Moon to demonstrate the trajectory and energy plots.

In [ ]:
earth = Body("Earth", 5.972e24, np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0]), "blue")
moon = Body("Moon", 7.348e22, np.array([384.4e6, 0.0, 0.0]), np.array([0.0, 1022.0, 0.0]), "gray")

sim = OrbitSimulator([earth, moon])
sim.run(duration=2592000, dt=3600)

plt.figure(figsize=(8, 8))
for name, data in sim.history.items():
    pos = np.array(data['pos'])
    plt.plot(pos[:, 0], pos[:, 1], label=name)
plt.title("Orbital Trajectories")
plt.xlabel("X (meters)")
plt.ylabel("Y (meters)")
plt.legend()
plt.axis('equal')
plt.show()

## Velocity and Energy Analysis
To ensure the simulation is physically accurate, we track the velocity magnitudes and the total mechanical energy of the system.

In [ ]:
def plot_physics(sim):
    steps = len(next(iter(sim.history.values()))['pos'])
    time_axis = np.arange(steps) * 3600 / 86400

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    for name, data in sim.history.items():
        ax1.plot(time_axis, data['vel'], label=f"{name} Velocity")
    ax1.set_title("Velocity over Time")
    ax1.set_xlabel("Time (days)")
    ax1.set_ylabel("Velocity (m/s)")
    ax1.legend()

    total_energy = np.zeros(steps)
    for i in range(steps):
        ke = 0
        for b_idx, b in enumerate(sim.bodies):
            v = sim.history[b.name]['vel'][i]
            ke += 0.5 * b.mass * v**2

        p1 = sim.history[sim.bodies[0].name]['pos'][i]
        p2 = sim.history[sim.bodies[1].name]['pos'][i]
        dist = np.linalg.norm(p1 - p2)
        pe = -G * sim.bodies[0].mass * sim.bodies[1].mass / dist
        total_energy[i] = ke + pe

    ax2.plot(time_axis, total_energy, color='red')
    ax2.set_title("Total System Energy (Conservation Check)")
    ax2.set_xlabel("Time (days)")
    ax2.set_ylabel("Energy (Joules)")
    plt.tight_layout()
    plt.show()

plot_physics(sim)

## Binary Star and N-Body Scenarios
Below we define a stable binary star system and a chaotic 3-body system to demonstrate the flexibility of the OrbitLab engine.

In [ ]:
star1 = Body("Star A", 2.0e30, np.array([-1.5e11, 0, 0]), np.array([0, 15000, 0]), "orange")
star2 = Body("Star B", 2.0e30, np.array([1.5e11, 0, 0]), np.array([0, -15000, 0]), "yellow")

binary_sim = OrbitSimulator([star1, star2])
binary_sim.run(duration=3.154e7, dt=86400)

plt.figure(figsize=(6, 6))
for name, data in binary_sim.history.items():
    pos = np.array(data['pos'])
    plt.plot(pos[:, 0], pos[:, 1], label=name)
plt.title("Binary Star Trajectories")
plt.legend()
plt.show()

## N-Body and Satellite Scenarios
Finally, we simulate a chaotic three-body system and a satellite in low Earth orbit.

In [ ]:
b1 = Body("Body 1", 1e30, np.array([1e11, 0, 0]), np.array([0, 10000, 0]), "red")
b2 = Body("Body 2", 1e30, np.array([-1e11, 0, 0]), np.array([0, -10000, 0]), "green")
b3 = Body("Body 3", 1e30, np.array([0, 1e11, 0]), np.array([-10000, 0, 0]), "blue")

nbody_sim = OrbitSimulator([b1, b2, b3])
nbody_sim.run(duration=3.154e7, dt=10000)

earth_focus = Body("Earth", 5.972e24, np.array([0,0,0]), np.array([0,0,0]))
satellite = Body("Satellite", 1000, np.array([7e6, 0, 0]), np.array([0, 7500, 0]))

sat_sim = OrbitSimulator([earth_focus, satellite])
sat_sim.run(duration=6000, dt=10)

plt.figure(figsize=(6, 6))
for name, data in nbody_sim.history.items():
    pos = np.array(data['pos'])
    plt.plot(pos[:, 0], pos[:, 1], label=name)
plt.title("Chaotic N-Body Trajectories")
plt.legend()
plt.show()